In [2]:
#pip install pymysql sqlalchemy pandas numpy scikit-learn tensorflow


In [3]:
import pandas as pd
from sqlalchemy import create_engine

# 1) DB 연결
DB_USER = "root"
DB_PW   = "1234"
DB_HOST = "localhost"
DB_NAME = "running_db"  

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PW}@{DB_HOST}/{DB_NAME}")

# 2) running_record 에서 필요한 컬럼만 가져오기
query = """
SELECT
    user_id,
    start_time,
    distance_km,
    pace_km,
    avg_heart_rate,
    TIME_TO_SEC(duration_time) AS duration_sec
FROM running_record
WHERE distance_km IS NOT NULL
ORDER BY user_id, start_time
"""
df = pd.read_sql(query, engine)

print(df.head())


   user_id          start_time  distance_km  pace_km  avg_heart_rate  \
0        1 2025-01-01 06:07:00          3.5      6.0           162.0   
1        1 2025-01-17 06:17:00         11.0      6.1           124.0   
2        1 2025-02-18 05:50:00         11.8      6.1           164.0   
3        1 2025-03-11 05:50:00         10.5      6.1           144.0   
4        1 2025-03-29 19:00:00         10.8      6.1           144.0   

   duration_sec  
0          1260  
1          4020  
2          4800  
3          4500  
4          4500  


In [4]:
# DB 전처리
import numpy as np

# 1) 결측치 간단 처리 (여기선 드랍)
df = df.dropna(subset=['distance_km', 'pace_km', 'avg_heart_rate', 'duration_sec'])

# 2) 시간 정렬(이미 ORDER BY 했지만 다시함)
df['start_time'] = pd.to_datetime(df['start_time'])
df = df.sort_values(['user_id', 'start_time'])

# 3) feature / target 분리 준비
feature_cols = ['distance_km', 'pace_km', 'avg_heart_rate', 'duration_sec']
target_col   = 'distance_km'


In [5]:
from sklearn.preprocessing import StandardScaler

SEQ_LEN = 7  # 개인별 최근 7회 러닝 기록으로 다음 거리 예측 / 기록 수의 중간값이 7

def build_sequences(df, seq_len=10):
    X_list = []
    Y_list = []

    # 유저별로 시퀀스를 만듦
    for uid, user_df in df.groupby('user_id'):
        user_features = user_df[feature_cols].values   # (T, 4)
        user_target   = user_df[target_col].values     # (T,)

        # 각 유저 내에서 시퀀스 쪼갬
        for i in range(len(user_features) - seq_len):
            X_seq = user_features[i:i+seq_len]      # (seq_len, 4)
            y_val = user_target[i+seq_len]         # seq_len 뒤의 distance

            X_list.append(X_seq)
            Y_list.append(y_val)

    X = np.array(X_list)  # (N, seq_len, 4)
    Y = np.array(Y_list)  # (N,)
    return X, Y

X_raw, Y_raw = build_sequences(df, SEQ_LEN)
print("X_raw shape:", X_raw.shape)
print("Y_raw shape:", Y_raw.shape)


X_raw shape: (399, 7, 4)
Y_raw shape: (399,)


In [6]:
# LSTM 입력 변수를 스케일링
N, T, F = X_raw.shape
X_2d = X_raw.reshape(-1, F)

scaler = StandardScaler()
X_2d_scaled = scaler.fit_transform(X_2d)

# 다시 (N, T, F)로 복원
X = X_2d_scaled.reshape(N, T, F)
Y = Y_raw


In [7]:
# 시퀀스 기준으로 train,test 분리
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, shuffle=True, random_state=42
)

print(X_train.shape, X_test.shape)


(319, 7, 4) (80, 7, 4)


In [8]:
# LSTM 모델 학습

import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(SEQ_LEN, len(feature_cols))),
    layers.LSTM(64, return_sequences=True), # 첫번째 유닛: 적정 유닛 64
    layers.LSTM(32), # 두번째 유닛: 
    layers.Dense(16, activation='relu'),
    layers.Dense(1)  # 다음 distance_km 예측
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

history = model.fit(
    X_train, Y_train,
    validation_data=(X_test, Y_test),
    epochs=50,
    batch_size=32,
    verbose=1
)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 7, 64)          │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,625 (119.63 KB)

 Trainable params: 30,625 (119.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - loss: 99.1826 - mae: 8.1407 - val_loss: 110.3197 - val_mae: 8.7018
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 94.6013 - mae: 7.8568 - val_loss: 102.9631 - val_mae: 8.2689
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 84.2794 - mae: 7.1312 - val_loss: 81.9380 - val_mae: 6.8757
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 59.2870 - mae: 5.4034 - val_loss: 46.1814 - val_mae: 4.6622
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 36.7647 - mae: 4.5419 - val_loss: 35.5296 - val_mae: 4.6162
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 34.4513 - mae: 4.7264 - val_loss: 34.8521 - val_mae: 4.5218
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 33.1759 - mae: 4.5732 - val_loss: 35.4803 - val_mae: 4.3866
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 32.7347 - mae: 4.4701 - val_loss: 35.3763 - val_mae: 4.3791
Epoch 9/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/

In [9]:
# 러닝 거리 예측

def predict_next_distance_for_user(user_id, df, scaler, model, seq_len=SEQ_LEN):
    user_df = df[df['user_id'] == user_id].sort_values('start_time')

    # 최근 seq_len개만 가져오기
    recent = user_df.iloc[-seq_len:]
    X_recent = recent[feature_cols].values  # (seq_len, 4)

    # 스케일링 (주의: train에서 fit한 scaler 사용)
    X_recent_scaled = scaler.transform(X_recent)  # (seq_len, 4)

    # LSTM 입력 형태로 변환: (1, seq_len, 4)
    X_input = X_recent_scaled.reshape(1, seq_len, len(feature_cols))

    # 예측
    y_pred = model.predict(X_input)[0, 0]  # 스칼라 값
    return y_pred

example_user_id = 1
next_distance = predict_next_distance_for_user(example_user_id, df, scaler, model)
print(f"user {example_user_id} 다음 운동 예상 거리: {next_distance:.2f} km")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step
user 1 다음 운동 예상 거리: 9.67 km


In [ ]:
# 목표 추천

def suggest_monthly_goal(user_id, df, scaler, model,
                         runs_per_week=1, weeks_per_month=3):
    next_dist = predict_next_distance_for_user(user_id, df, scaler, model)
    expected_runs = runs_per_week * weeks_per_month
    monthly_goal = next_dist * expected_runs
    return next_dist, monthly_goal

uid = 2
next_dist, monthly_goal = suggest_monthly_goal(uid, df, scaler, model)

print(f"[user {uid} 의 러닝 분석]")
print(f"다음 번 러닝 거리는 {next_dist:.2f} km 달리기!")
print(f"이번 달 러닝 목표 거리는 {monthly_goal:.1f} km 를 추천합니다.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
[user 2 의 러닝 분석]
다음 번 러닝 거리는 8.03 km 달리기!
이번 달 추천 러닝 목표 거리는 24.1 km 를 추천합니다.


In [2]:
!pip install pymysql sqlalchemy pandas numpy scikit-learn tensorflow

import pandas as pd
from sqlalchemy import create_engine

# 1) DB 연결
DB_USER = "root"
DB_PW   = "1234"
DB_HOST = "localhost"
DB_NAME = "running_db"  

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PW}@{DB_HOST}/{DB_NAME}")

# 2) running_record 에서 필요한 컬럼만 가져오기
query = """
SELECT
    user_id,
    start_time,
    distance_km,
    pace_km,
    avg_heart_rate,
    TIME_TO_SEC(duration_time) AS duration_sec
FROM running_record
WHERE distance_km IS NOT NULL
ORDER BY user_id, start_time
"""
df = pd.read_sql(query, engine)

print(df.head())

# DB 전처리
import numpy as np

# 1) 결측치 간단 처리 (여기선 드랍)
df = df.dropna(subset=['distance_km', 'pace_km', 'avg_heart_rate', 'duration_sec'])

# 2) 시간 정렬(이미 ORDER BY 했지만 다시함)
df['start_time'] = pd.to_datetime(df['start_time'])
df = df.sort_values(['user_id', 'start_time'])

# 3) feature / target 분리 준비
feature_cols = ['distance_km', 'pace_km', 'avg_heart_rate', 'duration_sec']
target_col   = 'distance_km'

from sklearn.preprocessing import StandardScaler

SEQ_LEN = 7  # 개인별 최근 7회 러닝 기록으로 다음 거리 예측 / 기록 수의 중간값이 7

def build_sequences(df, seq_len=10):
    X_list = []
    Y_list = []

    # 유저별로 시퀀스를 만듦
    for uid, user_df in df.groupby('user_id'):
        user_features = user_df[feature_cols].values   # (T, 4)
        user_target   = user_df[target_col].values     # (T,)

        # 각 유저 내에서 시퀀스 쪼갬
        for i in range(len(user_features) - seq_len):
            X_seq = user_features[i:i+seq_len]      # (seq_len, 4)
            y_val = user_target[i+seq_len]         # seq_len 뒤의 distance

            X_list.append(X_seq)
            Y_list.append(y_val)

    X = np.array(X_list)  # (N, seq_len, 4)
    Y = np.array(Y_list)  # (N,)
    return X, Y

X_raw, Y_raw = build_sequences(df, SEQ_LEN)
print("X_raw shape:", X_raw.shape)
print("Y_raw shape:", Y_raw.shape)

# LSTM 입력 변수를 스케일링
N, T, F = X_raw.shape
X_2d = X_raw.reshape(-1, F)

scaler = StandardScaler()
X_2d_scaled = scaler.fit_transform(X_2d)

# 다시 (N, T, F)로 복원
X = X_2d_scaled.reshape(N, T, F)
Y = Y_raw

# 시퀀스 기준으로 train,test 분리
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, shuffle=True, random_state=42
)

print(X_train.shape, X_test.shape)

# LSTM 모델 학습

import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(SEQ_LEN, len(feature_cols))),
    layers.LSTM(64, return_sequences=True), # 첫번째 유닛: 적정 유닛 64
    layers.LSTM(32), # 두번째 유닛: 
    layers.Dense(16, activation='relu'),
    layers.Dense(1)  # 다음 distance_km 예측
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

history = model.fit(
    X_train, Y_train,
    validation_data=(X_test, Y_test),
    epochs=50,
    batch_size=32,
    verbose=1
)

# 러닝 거리 예측

def predict_next_distance_for_user(user_id, df, scaler, model, seq_len=SEQ_LEN):
    user_df = df[df['user_id'] == user_id].sort_values('start_time')

    # 최근 seq_len개만 가져오기
    recent = user_df.iloc[-seq_len:]
    X_recent = recent[feature_cols].values  # (seq_len, 4)

    # 스케일링 (주의: train에서 fit한 scaler 사용)
    X_recent_scaled = scaler.transform(X_recent)  # (seq_len, 4)

    # LSTM 입력 형태로 변환: (1, seq_len, 4)
    X_input = X_recent_scaled.reshape(1, seq_len, len(feature_cols))

    # 예측
    y_pred = model.predict(X_input)[0, 0]  # 스칼라 값
    return y_pred

example_user_id = 1
next_distance = predict_next_distance_for_user(example_user_id, df, scaler, model)
print(f"user {example_user_id} 다음 운동 예상 거리: {next_distance:.2f} km")

# 목표 추천

def suggest_monthly_goal(user_id, df, scaler, model,
                         runs_per_week=1, weeks_per_month=3):
    next_dist = predict_next_distance_for_user(user_id, df, scaler, model)
    expected_runs = runs_per_week * weeks_per_month
    monthly_goal = next_dist * expected_runs
    return next_dist, monthly_goal

uid = 1
next_dist, monthly_goal = suggest_monthly_goal(uid, df, scaler, model)

print(f"[user {uid}]")
print(f"- 다음 운동 예상 거리: {next_dist:.2f} km")
print(f"- 이번 달 추천 목표 거리(예상): {monthly_goal:.1f} km")

Defaulting to user installation because normal site-packages is not writeable


OperationalError: (pymysql.err.OperationalError) (1049, "Unknown database 'running_db'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [1]:
model.save("lstm_distance.h5")
print("✔ 모델 저장 완료: lstm_distance.h5")


NameError: name 'model' is not defined